# 05 — Data Engineering Patterns with FastAPI

The patterns that actually come up when FastAPI sits in front of a data pipeline or model — file-upload ingestion, pagination over large result sets, streaming a large response instead of buffering it, background tasks for slow post-processing, and serving batch predictions.

In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*httpx2.*")   # silence a harmless TestClient/httpx notice

import io
import csv
from fastapi import FastAPI, UploadFile, File, BackgroundTasks, Query
from fastapi.testclient import TestClient
from fastapi.responses import StreamingResponse
from pydantic import BaseModel

app = FastAPI()

## 1. File upload for data ingestion

`UploadFile` streams the upload rather than loading the whole file into memory at once (unlike `bytes`, which reads it all up front) — the right choice for accepting a CSV/JSON upload that could be large. Validate/parse it immediately so bad input is rejected before anything downstream sees it.

In [2]:
@app.post("/ingest/csv")
async def ingest_csv(file: UploadFile = File(...)):
    contents = await file.read()
    reader = csv.DictReader(io.StringIO(contents.decode("utf-8")))
    rows = list(reader)
    if not rows:
        return {"error": "empty file", "rows_ingested": 0}
    return {"filename": file.filename, "rows_ingested": len(rows), "columns": list(rows[0].keys())}

client = TestClient(app)
csv_bytes = b"user_id,amount\n1,10.5\n2,20.0\n"
resp = client.post("/ingest/csv", files={"file": ("data.csv", csv_bytes, "text/csv")})
print(resp.json())

{'filename': 'data.csv', 'rows_ingested': 2, 'columns': ['user_id', 'amount']}


## 2. Pagination over large result sets

Never return an entire table in one response. The two standard shapes:

- **Offset pagination** (`skip`/`limit`) — simple, but `OFFSET` gets slower on large tables as `skip` grows (the DB still scans past the skipped rows), and results can shift if rows are inserted/deleted between pages.
- **Cursor pagination** (`after_id` / an opaque cursor token) — the client passes back the last-seen key, and the query becomes `WHERE id > after_id LIMIT n`, which stays fast regardless of how deep into the dataset you are and is stable under concurrent writes. Preferred for large or frequently-changing datasets.

In [3]:
FAKE_TABLE = [{"id": i, "value": f"row-{i}"} for i in range(1, 101)]

@app.get("/records-offset")
def list_records_offset(skip: int = 0, limit: int = Query(default=10, le=50)):
    page = FAKE_TABLE[skip: skip + limit]
    return {"items": page, "skip": skip, "limit": limit, "total": len(FAKE_TABLE)}

@app.get("/records-cursor")
def list_records_cursor(after_id: int = 0, limit: int = Query(default=10, le=50)):
    page = [r for r in FAKE_TABLE if r["id"] > after_id][:limit]
    next_cursor = page[-1]["id"] if page else None
    return {"items": page, "next_cursor": next_cursor}

client = TestClient(app)
print(client.get("/records-offset?skip=95&limit=10").json())

first_page = client.get("/records-cursor?limit=3").json()
print(first_page)
second_page = client.get(f"/records-cursor?after_id={first_page['next_cursor']}&limit=3").json()
print(second_page)

{'items': [{'id': 96, 'value': 'row-96'}, {'id': 97, 'value': 'row-97'}, {'id': 98, 'value': 'row-98'}, {'id': 99, 'value': 'row-99'}, {'id': 100, 'value': 'row-100'}], 'skip': 95, 'limit': 10, 'total': 100}
{'items': [{'id': 1, 'value': 'row-1'}, {'id': 2, 'value': 'row-2'}, {'id': 3, 'value': 'row-3'}], 'next_cursor': 3}
{'items': [{'id': 4, 'value': 'row-4'}, {'id': 5, 'value': 'row-5'}, {'id': 6, 'value': 'row-6'}], 'next_cursor': 6}


## 3. Streaming a large response

`StreamingResponse` sends data as it's generated instead of building the entire response body in memory first — the right tool for exporting a large query result as CSV/NDJSON without spiking memory on either the server or a naive all-at-once client.

In [4]:
def generate_csv_rows():
    yield "id,value\n"
    for row in FAKE_TABLE:
        yield f"{row['id']},{row['value']}\n"

@app.get("/export.csv")
def export_csv():
    return StreamingResponse(generate_csv_rows(), media_type="text/csv")

client = TestClient(app)
resp = client.get("/export.csv")
print(resp.headers["content-type"])
print(resp.text[:60])
print("total lines:", len(resp.text.strip().split(chr(10))))

text/csv; charset=utf-8
id,value
1,row-1
2,row-2
3,row-3
4,row-4
5,row-5
6,row-6
7,r
total lines: 101


## 4. Background tasks for slow post-processing

`BackgroundTasks` runs a function **after** the response has already been sent — right for lightweight fire-and-forget work (writing an audit log, sending a notification) directly attached to a request. For anything slow, retryable, or that must survive a server restart, the honest answer is a real task queue/broker (Celery, an SQS/Kafka-backed worker) — `BackgroundTasks` has no persistence or retry semantics of its own.

In [5]:
processed_log = []

def log_ingestion(filename: str, row_count: int):
    processed_log.append({"filename": filename, "row_count": row_count})

@app.post("/ingest/csv-with-logging")
async def ingest_csv_with_logging(background_tasks: BackgroundTasks, file: UploadFile = File(...)):
    contents = await file.read()
    rows = list(csv.DictReader(io.StringIO(contents.decode("utf-8"))))
    background_tasks.add_task(log_ingestion, file.filename, len(rows))
    return {"accepted": True, "rows": len(rows)}

client = TestClient(app)
resp = client.post("/ingest/csv-with-logging", files={"file": ("batch.csv", csv_bytes, "text/csv")})
print(resp.json())
print("background task ran:", processed_log)   # populated -- TestClient waits for background tasks

{'accepted': True, 'rows': 2}
background task ran: [{'filename': 'batch.csv', 'row_count': 2}]


## 5. Serving batch predictions

A common ML-serving shape: accept a list of feature rows, run them through a (here, fake) model, and return predictions aligned to input order. Validating the request as a list of a Pydantic model gets per-row shape checking for free — a malformed row anywhere in the batch is rejected before any prediction runs, rather than failing halfway through.

In [6]:
class PredictionRequest(BaseModel):
    features: list[float]

class PredictionResponse(BaseModel):
    prediction: float
    model_version: str

def fake_model_predict(features: list[float]) -> float:
    return sum(features) / len(features)   # stand-in for a real model.predict()

@app.post("/predict/batch", response_model=list[PredictionResponse])
def predict_batch(requests: list[PredictionRequest]):
    return [
        PredictionResponse(prediction=fake_model_predict(r.features), model_version="v1")
        for r in requests
    ]

client = TestClient(app)
payload = [{"features": [1.0, 2.0, 3.0]}, {"features": [10.0, 20.0]}]
print(client.post("/predict/batch", json=payload).json())

[{'prediction': 2.0, 'model_version': 'v1'}, {'prediction': 15.0, 'model_version': 'v1'}]


## 6. Interview Q&A

1. **"Why `UploadFile` instead of just reading `bytes` from the request body?"** — `UploadFile` streams from a spooled temp file rather than loading the entire upload into memory immediately, which matters once uploads can be large.
2. **"Offset vs. cursor pagination — when would you use each?"** — offset (`skip`/`limit`) is simplest for small/stable datasets; cursor-based pagination (`WHERE id > after_id`) stays fast at any depth and is stable under concurrent inserts/deletes, so it's preferred for large or actively-written tables.
3. **"Would you use `BackgroundTasks` to send data to a slow downstream service?"** — only if losing that work on a crash/restart is acceptable; anything that needs retries or durability belongs in a real task queue (Celery, SQS-backed workers), not `BackgroundTasks`.
4. **"How would you export a 5 GB query result over HTTP without running the server out of memory?"** — `StreamingResponse` over a generator that yields rows incrementally (e.g. reading from a DB cursor in batches), never materializing the whole result as one Python object.

## Summary

- `UploadFile` for large uploads; validate/parse immediately so bad input never reaches a pipeline.
- Prefer cursor-based pagination over offset pagination once a table is large or frequently written to.
- `StreamingResponse` avoids buffering a large response body in memory; `BackgroundTasks` is for cheap, best-effort, non-durable post-response work only.
- Validating a request body as `list[Model]` gets per-row shape checking for a batch-scoring endpoint for free.
- Next: `06_testing_and_interview_problems.ipynb`.